# §52: DoTA **train** frames from CAP-DATA — pack + pixel gate (CPU)

Rule: `core/docs/final/KIM_RUN.md` **§52**. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. No model runs here.

The frozen map `core/splits/dota_train/DOTA_TRAIN_CAP.json` gives **3,023 of 3,275** DoTA train clips a CAP video (found from file-name order and frame counts; 2-fold check on val pairs: 1,208 right, 9 wrong). This notebook streams those CAP videos once, keeps one `{dota_id}.zip` per clip, and draws DoTA's boxes on the frames so you can judge whether the map is right **before any GPU is spent**.

| step | what | time (estimate) |
|---|---|---|
| 1 | setup, preflight | 1 min |
| 2 | tests on the touched code — **raises if red** | 1 min |
| 3 | **pack**: per CAP group, stream the tar parts. Resumable: re-run after a disconnect | several hours |
| 4 | finalize + verify -> `frames_manifest.json` | 10-30 min |
| 5 | box sheet: 76 numbered rows, unlabelled | 2 min |
| 6 | you type the rows that look wrong -> verdict. **Paste its output to Claude** | you |

**CPU runtime.** Needs >= 24 GB free on `/content` and about **57 GB** on Drive. Nothing is deleted. Never reads a val frame except the 6 control rows of step 5 (from `DoTA_CAP_frames`, not scored).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: CAP tar parts, DoTA annotations
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'
os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
for rel in ('core/tools/dota_train_match.py', 'core/tools/dota_train_sheet.py', 'core/splits/dota_train/DOTA_TRAIN_CAP.json'):
    assert (REPO / rel).is_file(), f'no {rel} under {REPO} -- re-upload kat-vad/'
assert '## 52. ' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §52 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

RAW         = constants.DATA_ROOT / 'MMAU' / 'raw' / 'CAP-DATA_chunks'
GROUPS      = {'1-10': (1, 10), '11': (11, 11), '12-42': (12, 42), '43': (43, 43), '44-62': (44, 62)}  # CAP type ranges
ARCHIVE     = Path(DRIVE_F) / 'data' / constants.DOTA_TRAIN_FRAMES_DIRNAME
VAL_ARCHIVE = Path(DRIVE_F) / 'data' / constants.DOTA_CAP_FRAMES_DIRNAME      # read-only: the 6 control rows
REPORTS     = Path(DRIVE_F) / 'outputs' / 'REPORTS' / constants.DOTA_TRAIN_OUTPUT_DIRNAME
LOCAL       = Path('/content/dota_train')
BATCH_GB    = constants.MMAU_STREAM_BATCH_GB
for p in (ARCHIVE, REPORTS, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
import hashlib  # noqa: E402
import urllib.request  # noqa: E402

ANN = LOCAL / 'DoTA_annotations.zip'              # DoTA's official annotation zip (10 MB)
ANN_URL = 'https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip'
if not ANN.exists():
    urllib.request.urlretrieve(ANN_URL, ANN)
frozen = json.loads((REPO / 'core/splits/dota_train/DOTA_TRAIN_CAP.json').read_text())
got_sha = hashlib.sha256(ANN.read_bytes()).hexdigest()
assert got_sha == frozen['sources']['annotations_sha256'], f'annotation zip differs from the one the map was built on: {got_sha}'
for g in GROUPS:
    n = len(sorted((RAW / g).glob(f'{g}.part_*')))
    assert n, f'no CAP parts under {RAW / g} -- deleted? tell Claude before re-downloading 267 GB'
    print(f'group {g}: {n} parts')
assert list(VAL_ARCHIVE.glob('*.zip')), f'no val archive at {VAL_ARCHIVE} (step 5 needs its control rows)'
free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f}'
print('archive ->', ARCHIVE, f'| /content free {free_gb:.0f} GB | annotations {ANN}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av
python -c "import av; print('deps OK')"

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd, log=None):
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}')


run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_train.py'])
ALIGNMENT = LOCAL / 'dota_train_alignment.json'
run([sys.executable, '-m', 'core.tools.dota_train_match', 'expand', '--out', ALIGNMENT])
clips = json.loads(ALIGNMENT.read_text())['clips']
assert len(clips) == 3023, len(clips)
group_of = {k: next(g for g, (lo, hi) in GROUPS.items() if lo <= c['cap_type'] <= hi) for k, c in clips.items()}


def settled(group):
    '''Clips this group is done with: packed and recorded, or a recorded frame-count mismatch.'''
    path = REPORTS / f'pack_{group}.json'
    entries = json.loads(path.read_text()) if path.exists() else {}
    on_drive = {p.stem for p in ARCHIVE.glob('*.zip')}
    return {k for k, v in entries.items() if v['reason'] == 'cap_frames_mismatch' or (v['reason'] == 'ok' and k in on_drive)}


for g in GROUPS:
    mine = {k for k in clips if group_of[k] == g}
    print(f'group {g}: {len(settled(g) & mine)} / {len(mine)} settled')

## 3. Pack (CPU, resumable). **Re-run this cell after a disconnect.**

Each group's tar is one stream and stops at the last CAP video it needs. A clip already on Drive is not streamed again.

In [ ]:
for g in GROUPS:
    done = settled(g)
    todo = {k: c for k, c in clips.items() if group_of[k] == g and k not in done}
    print(f'group {g}: {len(todo)} clips to stream')
    if not todo:
        continue
    part = LOCAL / f'alignment_{g}.json'          # only what is missing: pack then opens no finished zip on Drive
    part.write_text(json.dumps({'clips': todo}))
    t0 = time.time()
    run([sys.executable, '-m', 'core.tools.dota_cap_frames', 'pack', '--alignment', part,
         '--parts', *sorted((RAW / g).glob(f'{g}.part_*')), '--work-dir', LOCAL / 'work' / g,
         '--archive-dir', ARCHIVE, '--max-batch-gb', BATCH_GB, '--report', REPORTS / f'pack_{g}.json'],
        log=LOCAL / 'pack.log')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

## 4. Finalize + verify

Drops only the clips whose streamed CAP video did not have the annotated frame count; any other missing clip raises (re-run step 3). `verify` opens every zip.

In [ ]:
FINAL = REPORTS / 'dota_train_alignment_final.json'
if not (ARCHIVE / constants.DOTA_CAP_FRAMES_MANIFEST).exists():
    run([sys.executable, '-m', 'core.tools.dota_train_match', 'finalize', '--alignment', ALIGNMENT,
         '--archive-dir', ARCHIVE, '--reports', *[REPORTS / f'pack_{g}.json' for g in GROUPS], '--out', FINAL])
    run([sys.executable, '-m', 'core.tools.dota_cap_frames', 'verify', '--alignment', FINAL,
         '--archive-dir', ARCHIVE, '--report', REPORTS / 'verify.json'])
v = json.loads((REPORTS / 'verify.json').read_text())
final = json.loads(FINAL.read_text())
print({k: v[k] for k in ('clips', 'ok', 'frames', 'gb', 'pass')}, '| dropped', len(final['dropped']), final['dropped'][:10])
assert v['pass'], v['failed']

## 5. Box sheet — **judge every row by eye; do not open `key.json`**

Each row is one clip at the first, middle and last boxed frame of its accident. **Right:** the red box sits on a vehicle (or person) in all three frames. **Wrong:** a box on the sky, the road, or empty air. Judge the row, not one corner: DoTA's boxes are loose and a vehicle can be half hidden. Some rows are wrong **on purpose**. A row is wrong when **at least one of its three frames** has no participant under the box.

In [ ]:
SHEET = REPORTS / 'box_sheet'
if not (SHEET / 'key.json').exists():
    run([sys.executable, '-m', 'core.tools.dota_train_sheet', 'sheet', '--alignment', FINAL, '--archive-dir', ARCHIVE,
         '--val-archive-dir', VAL_ARCHIVE, '--annotations', ANN, '--out-dir', SHEET])
from IPython.display import Image as Show, display  # noqa: E402

for page in sorted(SHEET.glob('page_*.png')):
    print(page.name)
    display(Show(filename=str(page)))

## 6. Type the numbers of the wrong rows, run once, **paste the output to Claude**

In [ ]:
FLAGGED = []            # e.g. [3, 17, 42]
CONFIRM = False         # set True when the list is final: scoring reveals the answer, so judge first
assert CONFIRM, 'fill FLAGGED, then set CONFIRM = True'
run([sys.executable, '-m', 'core.tools.dota_train_sheet', 'score', '--key', SHEET / 'key.json',
     '--flagged', *FLAGGED, '--out', REPORTS / 'box_gate.json'])
gate = json.loads((REPORTS / 'box_gate.json').read_text())
print(json.dumps(gate, indent=1))
print('frame sizes', json.loads((SHEET / 'key.json').read_text())['frame_sizes'])
drive.flush_and_unmount()